# Handling Imbalanced Data in AutoGluon Tabular

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](ADD_COLAB_LINK_HERE)
[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/images/badges/uramente_sagemaker_studiolab_badge.svg)](ADD_SAGEMAKER_LINK_HERE)

This tutorial demonstrates how to effectively handle imbalanced datasets when training models with AutoGluon Tabular. 

**What is Imbalanced Data?**

Imbalanced data refers to a situation in classification tasks where the classes are not represented equally. For example, in a binary classification problem, one class might have significantly more samples than the other. This is common in many real-world scenarios, such as:

* **Fraud detection:** Most transactions are legitimate, while fraudulent ones are rare.
* **Medical diagnosis:** The majority of patients might be healthy, with a small percentage having a particular disease.
* **Spam detection:** Most emails are not spam, but a small fraction are.

**Why is Imbalanced Data a Problem?**

Machine learning models trained on imbalanced datasets can be biased towards the majority class. They might achieve high accuracy by simply predicting the majority class for all instances, while performing poorly on the minority class, which is often the class of interest.

**Dataset: Credit Card Fraud Detection**

In this tutorial, we will use a credit card fraud detection dataset. This dataset is highly imbalanced, with a very small percentage of fraudulent transactions. The goal is to train a model that can accurately identify these fraudulent transactions.

**Dataset Download:**

The dataset can be downloaded from Kaggle: [Credit Card Fraud Detection](https://www.kaggle.com/mlg-ulb/creditcardfraud). You will need to download `archive.zip`, extract `creditcard.csv` and place it in the same directory as this notebook or provide the correct path.

In [ ]:
# Install AutoGluon and other necessary libraries
!pip install -U pip
!pip install -U autogluon
!pip install -U pandas scikit-learn requests opendatasets imbalanced-learn

## 1. Demonstrating the Problem with Imbalanced Data

First, let's load the credit card fraud dataset and examine the class distribution. We'll use the `opendatasets` library to download the data directly from Kaggle. You'll be prompted to enter your Kaggle username and API key. Make sure you have accepted the competition rules on the Kaggle page first.

In [ ]:
import pandas as pd
import opendatasets as od
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularDataset, TabularPredictor

# Download the dataset from Kaggle
dataset_url = 'https://www.kaggle.com/mlg-ulb/creditcardfraud'
od.download(dataset_url, force=False)

data_path = './creditcardfraud/creditcard.csv'
try:
    df = pd.read_csv(data_path)
except FileNotFoundError:
    print(f"Error: Dataset file not found at {data_path}.")
    raise

print(f"Shape of the dataframe: {df.shape}")
print("\nClass distribution:")
print(df['Class'].value_counts(normalize=True))

# Prepare data for AutoGluon
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Class'])

subsample_size = 20000
if len(train_df) > subsample_size:
    if train_df['Class'].nunique() > 1 and train_df['Class'].value_counts().min() > 1:
        train_small_df, _ = train_test_split(train_df, train_size=subsample_size, random_state=42, stratify=train_df['Class'])
    else:
        train_small_df = train_df.sample(n=subsample_size, random_state=42)
else:
    train_small_df = train_df

print(f"\nSubsampled training data shape: {train_small_df.shape}")
print("Subsampled training data class distribution:")
print(train_small_df['Class'].value_counts(normalize=True))

label = 'Class'
predictor_baseline = TabularPredictor(label=label, eval_metric='accuracy', path='ag_baseline')
predictor_baseline.fit(train_small_df.copy(), time_limit=60)

print("\nBaseline Predictor Evaluation (eval_metric='accuracy'):")
leaderboard_baseline = predictor_baseline.leaderboard(test_df.copy(), silent=True)
print(leaderboard_baseline)
eval_results_baseline = predictor_baseline.evaluate(test_df.copy(), silent=True)
print("\nBaseline Evaluation results:")
for metric, value in eval_results_baseline.items():
    print(f"{metric}: {value}")
print("\nBaseline Confusion Matrix:")
print(predictor_baseline.confusion_matrix(test_df.copy()))

As observed, the accuracy is very high, but the confusion matrix likely shows poor performance in identifying the minority class (fraudulent transactions). This is where choosing appropriate metrics becomes crucial.

## 2. Techniques for Handling Imbalanced Data

AutoGluon provides several ways to address imbalanced data. We'll explore a few key techniques.

### 2.1. Appropriate Evaluation Metrics

When dealing with imbalanced datasets, accuracy can be misleading. A model that always predicts the majority class might have high accuracy but will be useless for the task (e.g., failing to detect any fraud). More informative metrics include:

*   **Precision:** Out of all instances predicted as positive, how many were actually positive? (TP / (TP + FP))
*   **Recall (Sensitivity):** Out of all actual positive instances, how many were correctly predicted as positive? (TP / (TP + FN))
*   **F1-score:** The harmonic mean of precision and recall. It provides a balance between the two. (2 * (Precision * Recall) / (Precision + Recall))
*   **AUC-PR (Area Under the Precision-Recall Curve):** Similar to ROC AUC, but focuses on the trade-off between precision and recall across different thresholds. It's particularly useful for imbalanced classes.
*   **ROC AUC (Area Under the Receiver Operating Characteristic Curve):** Measures the ability of the model to distinguish between classes. While generally good, for highly imbalanced data, AUC-PR is often preferred.

AutoGluon allows you to specify the `eval_metric` during `TabularPredictor` initialization. For imbalanced binary classification, `f1`, `average_precision` (for AUC-PR), or `roc_auc` are good choices. Let's train a predictor with `f1` as the evaluation metric.

In [ ]:
predictor_f1 = TabularPredictor(label=label, eval_metric='f1', path='ag_f1_metric')

# Fit the predictor on the same subsampled training data
predictor_f1.fit(train_small_df.copy(), time_limit=60)

print("\nF1 Predictor Evaluation (eval_metric='f1'):")
leaderboard_f1 = predictor_f1.leaderboard(test_df.copy(), silent=True)
print(leaderboard_f1)

eval_results_f1 = predictor_f1.evaluate(test_df.copy(), silent=True)
print("\nF1 Evaluation results:")
for metric, value in eval_results_f1.items():
    print(f"{metric}: {value}")

print("\nF1 Confusion Matrix:")
print(predictor_f1.confusion_matrix(test_df.copy()))

print("\nComparison with Baseline (Accuracy-optimized):")
print(f"Baseline (Accuracy) F1: {eval_results_baseline.get('f1', 'N/A')}")
print(f"New (F1-optimized) F1: {eval_results_f1.get('f1', 'N/A')}")
print(f"Baseline (Accuracy) Recall: {eval_results_baseline.get('recall', 'N/A')}")
print(f"New (F1-optimized) Recall: {eval_results_f1.get('recall', 'N/A')}")
print(f"Baseline (Accuracy) Precision: {eval_results_baseline.get('precision', 'N/A')}")
print(f"New (F1-optimized) Precision: {eval_results_f1.get('precision', 'N/A')}")

By optimizing for F1-score, the model now likely does a better job at identifying the minority class (frauds), which should be reflected in higher recall and a better balance between precision and recall, even if the overall accuracy might be slightly lower than the baseline.

### 2.2. Class Weighting (`sample_weight`)

Class weighting is a technique where you assign different weights to different classes during training. The idea is to give a higher weight to the minority class, forcing the model to pay more attention to it and penalizing errors on the minority class more heavily.

AutoGluon's `TabularPredictor` supports instance weighting through the `sample_weight` parameter in the `fit()` method. You need to provide a column in your training data that contains the weight for each sample.

A common way to calculate weights is to use the inverse of the class frequencies. For a binary classification problem:
*   Weight for class 0 = Total Samples / (2 * Number of Class 0 Samples)
*   Weight for class 1 = Total Samples / (2 * Number of Class 1 Samples)

Let's calculate these weights for our training data and train a new predictor.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

# Use the subsampled training data for calculating weights
y_train = train_small_df[label]

# Calculate sample weights using sklearn's utility
# This assigns weights inversely proportional to class frequencies
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Add the weights as a new column to the training DataFrame
train_weighted_df = train_small_df.copy()
train_weighted_df['sample_weight_col'] = sample_weights

print("Sample weights for a few instances:")
print(train_weighted_df[[label, 'sample_weight_col']].head())
print(f"\nUnique weights for class 0: {train_weighted_df[train_weighted_df[label]==0]['sample_weight_col'].unique()}")
print(f"Unique weights for class 1: {train_weighted_df[train_weighted_df[label]==1]['sample_weight_col'].unique()}")

# Train a new predictor with sample weights
# We'll continue to use 'f1' as the eval_metric for fair comparison
predictor_weighted = TabularPredictor(
    label=label, 
    eval_metric='f1', 
    path='ag_weighted'
)

predictor_weighted.fit(
    train_weighted_df, 
    time_limit=60, 
    sample_weight='sample_weight_col' # Specify the column containing sample weights
)

print("\nWeighted Predictor Evaluation (eval_metric='f1', with sample_weight):")
leaderboard_weighted = predictor_weighted.leaderboard(test_df.copy(), silent=True)
print(leaderboard_weighted)

eval_results_weighted = predictor_weighted.evaluate(test_df.copy(), silent=True)
print("\nWeighted Evaluation results:")
for metric, value in eval_results_weighted.items():
    print(f"{metric}: {value}")

print("\nWeighted Confusion Matrix:")
print(predictor_weighted.confusion_matrix(test_df.copy()))

print("\nComparison with F1-optimized (no weights):")
print(f"F1-optimized (no weights) F1: {eval_results_f1.get('f1', 'N/A')}")
print(f"Weighted F1: {eval_results_weighted.get('f1', 'N/A')}")
print(f"F1-optimized (no weights) Recall: {eval_results_f1.get('recall', 'N/A')}")
print(f"Weighted Recall: {eval_results_weighted.get('recall', 'N/A')}")
print(f"F1-optimized (no weights) Precision: {eval_results_f1.get('precision', 'N/A')}")
print(f"Weighted Precision: {eval_results_weighted.get('precision', 'N/A')}")

Using `sample_weight` can often lead to improved performance on the minority class, as the model is explicitly told to pay more attention to these instances. The impact will vary depending on the dataset and models used by AutoGluon.

### 2.3. Other Techniques (Brief Discussion)

Besides choosing appropriate evaluation metrics and using class weights, other common techniques for handling imbalanced data include resampling the data:

*   **Oversampling the Minority Class:** This involves creating synthetic samples of the minority class. A popular algorithm for this is **SMOTE (Synthetic Minority Over-sampling Technique)**.
*   **Undersampling the Majority Class:** This involves removing samples from the majority class to make it more balanced with the minority class. This can be effective but may lead to loss of information.
*   **Combination of Oversampling and Undersampling:** Techniques like SMOTETomek or SMOTEENN combine both approaches.

These techniques are typically applied to the training data *before* passing it to `TabularPredictor`. Libraries like `imbalanced-learn` (which we installed earlier) provide implementations for these methods.

**Example (Conceptual):**
```python
# from imblearn.over_sampling import SMOTE
# from autogluon.tabular import TabularDataset

# # Assuming train_df is your original training data (features + label)
# X_train = train_df.drop(columns=[label])
# y_train = train_df[label]

# smote = SMOTE(random_state=42)
# X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# # Combine resampled features and labels back into a DataFrame
# train_resampled_df = pd.DataFrame(X_train_resampled, columns=X_train.columns)
# train_resampled_df[label] = y_train_resampled

# # Convert to TabularDataset if needed by older AutoGluon versions or for specific functionalities
# # train_ag_resampled_data = TabularDataset(train_resampled_df)

# # Then fit AutoGluon on train_resampled_df
# predictor_smote = TabularPredictor(label=label, eval_metric='f1', path='ag_smote')
# predictor_smote.fit(train_resampled_df, time_limit=60) 
# # Evaluate on the original (unmodified) test_df
# results = predictor_smote.evaluate(test_df)
```
While AutoGluon's built-in models and weighting options are powerful, for extremely imbalanced datasets or when specific resampling strategies are desired, pre-processing your data with `imbalanced-learn` can be a valuable step.

## 3. Conclusion

Working with imbalanced datasets presents unique challenges, primarily the risk of models that perform well on the majority class but poorly on the minority class, which is often the class of interest. Standard metrics like accuracy can be misleading in such scenarios.

This tutorial demonstrated several key techniques for addressing these challenges effectively using AutoGluon Tabular:

*   **Choosing Appropriate Evaluation Metrics:** We saw how selecting metrics like **F1-score**, **AUC-PR (average_precision)**, or **recall** instead of plain accuracy provides a much better assessment of model performance on imbalanced data and helps AutoGluon optimize for the minority class.
*   **Using `sample_weight`:** By assigning higher weights to minority class instances (e.g., using `compute_sample_weight` from `sklearn.utils.class_weight` and passing it to `TabularPredictor.fit()`), we can guide the learning process to pay more attention to these crucial examples.

We also briefly discussed that other methods, such as **resampling techniques** (oversampling the minority class with SMOTE, or undersampling the majority class) using libraries like `imbalanced-learn`, can be applied to preprocess the data *before* training with AutoGluon. These can be complementary to AutoGluon's built-in capabilities.

**Key Takeaways & Next Steps:**

1.  **Always inspect your class distribution.** If it's imbalanced, default accuracy is likely not the right metric.
2.  **Select an evaluation metric** that reflects your true objective (e.g., `f1`, `average_precision`, `recall`).
3.  **Experiment with `sample_weight`** in `TabularPredictor.fit()`.
4.  Consider **external resampling techniques** if needed for very extreme imbalances or specific requirements.
5.  **Evaluate multiple models and approaches.** The best strategy can be dataset-dependent.

By thoughtfully applying these techniques, you can significantly improve your model's ability to handle imbalanced data and deliver meaningful results.

**Further Resources:**

*   [AutoGluon Homepage](https://auto.gluon.ai/stable/index.html)
*   [`TabularPredictor` API Documentation](https://auto.gluon.ai/stable/api/autogluon.tabular.TabularPredictor.html)
*   [Tabular Essentials Tutorial](https://auto.gluon.ai/stable/tutorials/tabular_prediction/tabular-essentials.html)
*   [Tabular In-Depth Tutorial](https://auto.gluon.ai/stable/tutorials/tabular_prediction/tabular-indepth.html)
*   [Scikit-learn metrics for imbalanced classes](https://scikit-learn.org/stable/modules/model_evaluation.html#imbalanced-classification)
*   [Imbalanced-learn documentation](https://imbalanced-learn.org/stable/)